[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ricalanis/openenv-hackaton/blob/main/training/train_answering.ipynb)

# DataSage Stage 3: Answering GRPO Training

Pre-builds dataset with real environment observations (no `rollout_func`).
Reward functions replay the env with stored seeds for context-matched evaluation.
Includes Patronus Lynx integration for hallucination detection (optional).

**Fully self-contained** — runs in Colab with no repo clone or project imports.

**Requirements:** GPU (A100/H100), `HF_TOKEN`, `WANDB_API_KEY`.  
**Optional:** `PATRONUS_API_KEY` for Patronus Lynx.

In [1]:
# Setup: install dependencies
!pip install -q unsloth vllm trl datasets wandb requests
!pip install -q patronus  # optional

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.3/70.3 kB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 6.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.0/447.0 kB 25.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 432.9/432.9 MB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.6/192.6 kB 28.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 158.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 184.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 165.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.

In [2]:
# Load API keys from Colab Secrets or set manually
import os

try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
    # Optional: for Patronus Lynx hallucination detection
    os.environ["PATRONUS_API_KEY"] = userdata.get("PATRONUS_API_KEY")
    print("Loaded keys from Colab Secrets")
except Exception:
    pass  # Fill manually below if not using Colab Secrets

# Uncomment and fill if not using Colab Secrets:
# os.environ["HF_TOKEN"] = "hf_..."
# os.environ["WANDB_API_KEY"] = "wandb_..."
# os.environ["PATRONUS_API_KEY"] = "pat_..."  # optional

assert os.environ.get("HF_TOKEN"), "Set HF_TOKEN in Colab Secrets or above"
print("Keys loaded.")

Loaded keys from Colab Secrets
Keys loaded.


In [3]:
# Config + parser + reward helpers (all inlined, no project imports)
import json
import re
import requests
from dataclasses import dataclass, field

# ---- Config constants ----
WANDB_PROJECT = "datasage"
BASE_MODEL = "unsloth/Qwen2.5-3B-Instruct"
ENV_URL = "https://ricalanis-datasage-answering.hf.space"
HF_REPO = "ricalanis/datasage-qwen-answering"

LEARNING_RATE = 3e-6
NUM_TRAIN_EPOCHS = 3
PER_DEVICE_TRAIN_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 1
NUM_GENERATIONS = 8
MAX_COMPLETION_LENGTH = 256
MAX_PROMPT_LENGTH = 1024


# ---- Completion text extraction ----
def _get_text(completion) -> str:
    """Extract text from a completion (str or chat message list)."""
    if isinstance(completion, str):
        return completion
    if isinstance(completion, list):
        # Chat format: [{"role": "assistant", "content": "..."}]
        return completion[-1]["content"] if completion else ""
    return str(completion)


# ---- Parser ----
def parse_answering_action(text: str) -> dict:
    """Parse model output into an answering action dict."""
    json_match = re.search(r'\{[^{}]*"answer"[^{}]*\}', text, re.DOTALL)
    if json_match:
        try:
            data = json.loads(json_match.group())
            if "answer" in data:
                return data
        except json.JSONDecodeError:
            pass

    cited = re.findall(r'\b([A-Z][a-zA-Z]+(?:[A-Z][a-z]+)*)\b', text)
    return {
        "answer": text,
        "cited_columns": cited[:5],
        "reasoning": "",
    }


# ---- Persona definitions (inlined from environments/shared/personas.py) ----
@dataclass
class Persona:
    name: str
    role: str
    focus_areas: list[str]
    language_style: str
    keywords: list[str]
    anti_keywords: list[str]


PERSONAS = [
    Persona(
        name="Executive", role="executive",
        focus_areas=["costs", "ROI", "strategic risk", "portfolio trends", "year-over-year"],
        language_style="strategic-financial",
        keywords=["revenue", "cost", "ROI", "risk", "trend", "quarter",
                  "year-over-year", "impact", "budget", "margin", "growth"],
        anti_keywords=["I think", "maybe", "um", "idk"],
    ),
    Persona(
        name="Manager", role="manager",
        focus_areas=["team performance", "operational health", "process bottlenecks", "capacity"],
        language_style="operational-actionable",
        keywords=["team", "performance", "bottleneck", "capacity", "SLA",
                  "process", "action", "priority", "escalation", "delivery"],
        anti_keywords=["shareholder", "valuation", "IPO"],
    ),
    Persona(
        name="Individual Contributor", role="ic",
        focus_areas=["personal tasks", "deadlines", "what to do next", "simple explanations"],
        language_style="plain-personal",
        keywords=["my", "I should", "next step", "deadline", "help",
                  "understand", "priority", "task", "assigned"],
        anti_keywords=["KPI", "ROI", "portfolio", "strategic", "EBITDA"],
    ),
]


def _check_formality(text: str, style: str) -> float:
    """Check if text formality matches the expected language style."""
    text_lower = text.lower()
    if style == "strategic-financial":
        indicators = ["percent", "%", "million", "billion", "quarter", "fiscal",
                      "forecast", "benchmark", "metric"]
        hits = sum(1 for ind in indicators if ind in text_lower)
        return min(hits / 2.0, 1.0)
    elif style == "operational-actionable":
        indicators = ["action", "recommend", "should", "priority", "next steps",
                      "immediate", "plan", "schedule"]
        hits = sum(1 for ind in indicators if ind in text_lower)
        return min(hits / 2.0, 1.0)
    elif style == "plain-personal":
        sentences = text.split(".")
        avg_len = sum(len(s.split()) for s in sentences) / max(len(sentences), 1)
        return 1.0 if avg_len < 20 else max(0.0, 1.0 - (avg_len - 20) / 30)
    return 0.5


def score_persona_alignment(answer: str, persona: Persona) -> float:
    """Score how well an answer aligns with a persona's communication style."""
    answer_lower = answer.lower()
    words = re.findall(r'\w+', answer_lower)
    word_count = max(len(words), 1)

    keyword_hits = sum(1 for kw in persona.keywords if kw.lower() in answer_lower)
    keyword_score = min(keyword_hits / max(len(persona.keywords) * 0.3, 1), 1.0)

    anti_hits = sum(1 for akw in persona.anti_keywords if akw.lower() in answer_lower)
    anti_penalty = min(anti_hits * 0.15, 0.5)

    formality_score = _check_formality(answer, persona.language_style)

    raw_score = 0.50 * keyword_score + 0.20 * formality_score + 0.30 - anti_penalty
    return round(max(0.0, min(1.0, raw_score)), 4)


# ---- Reward helpers ----
_ANSWER_JSON_RE = re.compile(r'\{[^{}]*"answer"[^{}]*\}', re.DOTALL)


def persona_match_reward(completions, **kwargs) -> list[float]:
    """Reward alignment with the REQUESTED persona (not just any persona)."""
    persona_names = kwargs.get("persona_name", [])
    if not persona_names:
        return [0.0] * len(completions)
    persona_map = {p.name: p for p in PERSONAS}
    rewards = []
    for completion, p_name in zip(completions, persona_names):
        text = _get_text(completion)
        persona = persona_map.get(p_name)
        if persona:
            rewards.append(score_persona_alignment(text, persona))
        else:
            rewards.append(0.0)
    return rewards


def local_faithfulness_fn(completions, **kwargs) -> list[float]:
    """Local faithfulness heuristic: checks column citations and data grounding."""
    rewards = []
    for completion in completions:
        text = _get_text(completion)
        score = 0.0
        cited = re.findall(r'\b([A-Z][a-zA-Z]+(?:[A-Z][a-z]+)*)\b', text)
        if len(cited) >= 1:
            score += 0.3
        if len(cited) >= 3:
            score += 0.2
        if re.search(r'\d+\.?\d*%', text) or re.search(r'\b\d{2,}\b', text):
            score += 0.2
        if any(w in text.lower() for w in ["i believe", "probably", "might be",
                                            "i'm not sure", "i think maybe"]):
            score -= 0.2
        if len(text.strip()) < 50:
            score -= 0.3
        rewards.append(max(0.0, min(1.0, score)))
    return rewards


def patronus_reward_fn(completions, **kwargs) -> list[float]:
    """Use Patronus Lynx for hallucination detection, with local fallback."""
    api_key = os.environ.get("PATRONUS_API_KEY")
    if not api_key:
        return local_faithfulness_fn(completions, **kwargs)
    try:
        import patronus
        patronus.init()
        from patronus import Patronus, RemoteEvaluator
        client = Patronus()
        lynx = RemoteEvaluator("lynx", "patronus:hallucination")
        rewards = []
        for completion in completions:
            text = _get_text(completion)
            result = client.evaluate(
                evaluators=lynx,
                task_output=text,
                task_input="Answer the question based on the data.",
                task_context="",
            )
            rewards.append(float(result.results[0].score))
        return rewards
    except Exception:
        return local_faithfulness_fn(completions, **kwargs)


def answering_json_format_reward(completions, **kwargs) -> list[float]:
    """Reward well-formed JSON answer actions."""
    rewards = []
    for completion in completions:
        text = _get_text(completion)
        match = _ANSWER_JSON_RE.search(text)
        if match:
            try:
                data = json.loads(match.group())
                has_answer = bool(data.get("answer", "").strip())
                has_cited = bool(data.get("cited_columns"))
                has_reasoning = bool(data.get("reasoning", "").strip())
                if has_answer and has_cited and has_reasoning:
                    rewards.append(1.0)
                elif has_answer and has_cited:
                    rewards.append(0.7)
                elif has_answer:
                    rewards.append(0.4)
                else:
                    rewards.append(0.2)
            except (json.JSONDecodeError, AttributeError):
                rewards.append(0.2)
        else:
            rewards.append(0.0)
    return rewards


print(f"Environment URL: {ENV_URL}")
print(f"Base model: {BASE_MODEL}")

Environment URL: https://ricalanis-datasage-answering.hf.space
Base model: unsloth/Qwen2.5-3B-Instruct


In [4]:
# Model loading via Unsloth
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL, max_seq_length=2048, load_in_4bit=True,
    fast_inference=True, max_lora_rank=16, gpu_memory_utilization=0.6,
)
model = FastLanguageModel.get_peft_model(
    model, r=16, lora_alpha=16, lora_dropout=0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print(f"Model loaded: {BASE_MODEL}")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
INFO 03-08 10:21:20 [vllm_utils.py:723] Unsloth: Patching vLLM v1 graph capture
==((====))==  Unsloth 2026.3.3: Fast Qwen2 patching. Transformers: 4.57.6. vLLM: 0.17.0.
   \\   /|    NVIDIA RTX PRO 6000 Blackwell Server Edition. Num GPUs = 1. Max memory: 94.971 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: vLLM loading unsloth/qwen2.5-3b-instruct-unsloth-bnb-4bit with actual GPU utilization = 59.54%
Unsloth: Your GPU has CUDA compute capability 12.0 with VRAM = 94.97 GB.
Unsloth: Using conservativeness = 1.0. Chunked prefill tokens = 2048. Num Sequences = 128.
Unsloth: vLLM

/usr/local/lib/python3.12/dist-packages/pydantic/type_adapter.py:605: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `enum` - serialized value may not be as expected [field_name='mode', input_value=3, input_type=int])
  return self.serializer.to_python(


WARNING 03-08 10:21:28 [arg_utils.py:1321] The global random seed is set to 0. Since VLLM_ENABLE_V1_MULTIPROCESSING is set to False, this may affect the random state of the Python process that launched vLLM.
INFO 03-08 10:21:41 [model.py:531] Resolved architecture: Qwen2ForCausalLM
INFO 03-08 10:21:41 [model.py:1554] Using max model len 2048
INFO 03-08 10:21:41 [scheduler.py:231] Chunked prefill is enabled with max_num_batched_tokens=8192.
Unsloth: vLLM Bitsandbytes config using kwargs = {'load_in_8bit': False, 'load_in_4bit': True, 'bnb_4bit_compute_dtype': 'bfloat16', 'bnb_4bit_quant_storage': 'uint8', 'bnb_4bit_quant_type': 'nf4', 'bnb_4bit_use_double_quant': True, 'llm_int8_enable_fp32_cpu_offload': False, 'llm_int8_has_fp16_weight': False, 'llm_int8_skip_modules': ['lm_head', 'multi_modal_projector', 'merger', 'modality_projection', 'model.layers.2.mlp', 'model.layers.3.mlp', 'model.layers.30.mlp'], 'llm_int8_threshold': 6.0}
INFO 03-08 10:21:41 [vllm.py:747] Asynchronous scheduli

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

unsloth/qwen2.5-3b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Unsloth 2026.3.3 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


Model loaded: unsloth/Qwen2.5-3B-Instruct


In [5]:
# System prompt and task descriptions
SYSTEM_PROMPT = """\
You are a data-driven enterprise analyst. You answer questions about \
datasets across multiple domains (HR, Sales, Project Management, \
IT Operations) tailored to the audience persona.

Personas:
- Executive: Focus on costs, ROI, strategic risk, portfolio trends, year-over-year metrics.
- Manager: Focus on team performance, operational health, process bottlenecks, capacity.
- Individual Contributor: Focus on personal tasks, deadlines, what to do next.

Respond with a JSON answer:
{"answer": "<your answer>", "cited_columns": ["col1", "col2"], "reasoning": "<chain-of-thought>"}

Rules:
1. Ground every claim in the data.
2. Match your tone and vocabulary to the persona.
3. Be concise but thorough.
4. Never fabricate numbers."""

TASK_DESCRIPTIONS = [
    "Answer the question based on the data, tailored to the persona.",
    "Provide a data-grounded answer appropriate for this audience.",
    "Analyze the data and answer the question in the persona's style.",
    "Use the dataset to answer accurately, matching the persona's focus.",
    "Generate a faithful, persona-aligned answer citing real data.",
    "Answer using statistics from the data, in the right tone for this persona.",
    "Review the data summary and answer the question for this stakeholder.",
    "Craft a response grounded in the data that matches the persona's needs.",
]

In [6]:
# Dataset: pre-built with real environment observations
import random
from datasets import Dataset

random.seed(42)
SEEDS = [42 + i for i in range(64)]
prompts = []
persona_names = []

print("Building dataset with real environment observations...")
for i, seed in enumerate(SEEDS):
    task_desc = random.choice(TASK_DESCRIPTIONS)
    resp = requests.post(f"{ENV_URL}/web/reset", json={"seed": seed})
    resp.raise_for_status()
    obs = resp.json()["observation"]

    prompts.append([
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": (
            f"Domain: {obs['domain']}\nPersona: {obs['persona']}\n"
            f"Persona Focus: {obs['persona_description']}\n\n"
            f"Question: {obs['question']}\n\n"
            f"Dataset Summary:\n{obs['dataset_summary']}\n\n"
            f"Column Statistics:\n{obs['column_stats']}\n\n"
            f"Available Columns: {', '.join(obs['available_columns'])}\n\n"
            f"Task: {task_desc}"
        )},
    ])
    persona_names.append(obs["persona"])
    if (i + 1) % 16 == 0:
        print(f"  Built {i + 1}/64 examples")

dataset = Dataset.from_dict({
    "prompt": prompts,
    "seed": SEEDS,
    "persona_name": persona_names,
})
print(f"Dataset size: {len(dataset)}")

Building dataset with real environment observations...
  Built 16/64 examples
  Built 32/64 examples
  Built 48/64 examples
  Built 64/64 examples
Dataset size: 64


In [7]:
# Environment reward function (calls env directly with stored seeds)
def answering_env_reward(completions, **kwargs) -> list[float]:
    """Primary reward: replay env with stored seed, step with parsed action."""
    seeds = kwargs.get("seed", [0] * len(completions))
    rewards = []
    for completion, seed in zip(completions, seeds):
        try:
            text = _get_text(completion)
            action_dict = parse_answering_action(text)
            requests.post(f"{ENV_URL}/web/reset", json={"seed": int(seed)})
            resp = requests.post(f"{ENV_URL}/web/step", json={"action": action_dict})
            resp.raise_for_status()
            rewards.append(float(resp.json().get("reward", 0.0)))
        except Exception as e:
            print(f"Env error: {e}")
            rewards.append(0.0)
    return rewards

In [8]:
# GRPO training config
from trl import GRPOConfig

training_args = GRPOConfig(
    output_dir="./outputs/answering-grpo",
    use_vllm=True,
    learning_rate=LEARNING_RATE,
    adam_beta1=0.9,
    adam_beta2=0.99,
    weight_decay=0.1,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    optim="adamw_8bit",
    num_train_epochs=NUM_TRAIN_EPOCHS,
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    num_generations=NUM_GENERATIONS,
    max_completion_length=MAX_COMPLETION_LENGTH,
    max_prompt_length=MAX_PROMPT_LENGTH,
    max_grad_norm=0.1,
    logging_steps=1, save_steps=50,
    report_to="wandb", run_name="datasage-answering-grpo-v2",
)
os.environ.setdefault("WANDB_PROJECT", WANDB_PROJECT)

Unsloth: We now expect `per_device_train_batch_size` * `gradient_accumulation_steps` * `world_size` to be a multiple of `num_generations`.
We will change the batch size of 1 to the `num_generations` of 8


'datasage'

In [9]:
# Create trainer and train
from trl import GRPOTrainer

trainer = GRPOTrainer(
    model=model, processing_class=tokenizer, args=training_args,
    train_dataset=dataset,
    reward_funcs=[answering_env_reward, patronus_reward_fn, answering_json_format_reward, persona_match_reward],
)
print("Starting Stage 3 (Answering) GRPO training...")
trainer.train()

Starting Stage 3 (Answering) GRPO training...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 64 | Num Epochs = 3 | Total steps = 192
O^O/ \_/ \    Batch size per device = 8 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (8 x 1 x 1) = 8
 "-____-"     Trainable parameters = 29,933,568 of 3,115,872,256 (0.96% trained)
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: ricalanis to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: Detected [huggingface_hub.inference, openai] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai/


WARNING 03-08 10:24:19 [input_processor.py:168] vLLM has deprecated support for supporting different tokenizers for different LoRAs. By default, vLLM uses base model's tokenizer. If you are using a LoRA with its own tokenizer, consider specifying `--tokenizer [lora_path]` to use the LoRA tokenizer.
Unsloth: Will smartly offload gradients to save VRAM!
https://api.patronus.ai/v1/evaluators
https://api.patronus.ai/v1/evaluator-criteria
https://api.patronus.ai/v1/evaluate
https://api.patronus.ai/v1/evaluate
https://api.patronus.ai/v1/evaluate
https://api.patronus.ai/v1/evaluate
https://api.patronus.ai/v1/evaluate
https://api.patronus.ai/v1/evaluate
https://api.patronus.ai/v1/evaluate
https://api.patronus.ai/v1/evaluate


Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,sampling / sampling_logp_difference / mean,sampling / sampling_logp_difference / max,sampling / importance_sampling_ratio / min,sampling / importance_sampling_ratio / mean,sampling / importance_sampling_ratio / max,kl,rewards / answering_env_reward / mean,rewards / answering_env_reward / std,rewards / patronus_reward_fn / mean,rewards / patronus_reward_fn / std,rewards / answering_json_format_reward / mean,rewards / answering_json_format_reward / std,rewards / persona_match_reward / mean,rewards / persona_match_reward / std
1,0.029800,2.633462,0.369990,241.000000,218.000000,256.000000,0.375000,232.000000,218.000000,256.000000,0,0,0,0,0,-0.000000,0.412275,0.144232,1.000000,0.000000,0.500000,0.534522,0.721187,0.148893
2,-0.008600,1.819300,0.595167,240.750000,145.000000,256.000000,0.750000,195.000000,145.000000,245.000000,No Log,No Log,No Log,No Log,No Log,-0.000000,0.373863,0.079609,0.750000,0.462910,0.250000,0.462910,0.445437,0.098204
3,0.060100,2.222925,0.606233,206.250000,170.000000,256.000000,0.125000,199.142868,170.000000,238.000000,No Log,No Log,No Log,No Log,No Log,0.001357,0.289587,0.275716,0.500000,0.534522,0.675000,0.465219,0.758337,0.290454
4,0.080000,2.618600,0.666095,234.000000,176.000000,256.000000,0.500000,212.000000,176.000000,251.000000,No Log,No Log,No Log,No Log,No Log,0.001186,0.618562,0.117592,0.640000,0.497910,0.500000,0.534522,0.860038,0.156361
5,0.056100,2.261375,0.545111,242.125000,216.000000,256.000000,0.500000,228.250000,216.000000,241.000000,No Log,No Log,No Log,No Log,No Log,0.001082,0.348525,0.114526,0.876250,0.350018,0.500000,0.534522,0.536600,0.125571
6,0.034900,2.211662,0.497211,165.500000,100.000000,256.000000,0.125000,152.571442,100.000000,197.000000,No Log,No Log,No Log,No Log,No Log,0.000930,0.303338,0.195858,0.625000,0.517549,0.675000,0.465219,0.608325,0.274727
7,0.100800,2.217850,0.582587,201.375000,159.000000,256.000000,0.250000,183.166672,159.000000,199.000000,No Log,No Log,No Log,No Log,No Log,0.000925,0.374538,0.135505,0.635000,0.504154,0.750000,0.462910,0.458312,0.073718
8,0.072000,2.947100,0.659951,227.375000,162.000000,256.000000,0.375000,210.199997,162.000000,239.000000,No Log,No Log,No Log,No Log,No Log,0.001025,0.452075,0.118365,1.000000,0.000000,0.625000,0.517549,0.870025,0.128383
9,0.037800,2.307850,0.770200,217.250000,141.000000,256.000000,0.125000,211.714294,141.000000,245.000000,No Log,No Log,No Log,No Log,No Log,0.001185,0.311538,0.122946,0.751250,0.457522,0.875000,0.353553,0.370063,0.210192
10,0.015800,2.057312,0.830834,214.625000,162.000000,256.000000,0.125000,208.714294,162.000000,254.000000,No Log,No Log,No Log,No Log,No Log,0.001122,0.297100,0.160953,0.501250,0.533196,0.750000,0.462910,0.508963,0.072574


...

/tmp/ipykernel_5408/2897798421.py:180: UserWarning: The Patronus SDK has already been initialized. Duplicate initialization attempts are ignored.
  patronus.init()


https://api.patronus.ai/v1/evaluators
https://api.patronus.ai/v1/evaluator-criteria
https://api.patronus.ai/v1/evaluate


ERROR:patronus.sdk:Evaluator raised an exception: RetryError: execution failed after 1/3 attempts: UnrecoverableAPIError: Response with unexpected status code: 402: response content: {"detail":"Insufficient funds"}
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/patronus/retry.py", line 73, in wrapper
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/patronus/evals/evaluators.py", line 861, in _evaluate
    return self._get_api().evaluate_one_sync(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/patronus/api/api_client.py", line 174, in evaluate_one_sync
    return self._evaluate_one_process_resp(resp)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/patronus/api/api_client.py", line 218, in _evaluate_one_process_resp
    raise UnrecoverableAPIError(
patronus.api.api_client_base.UnrecoverableAPIE

profiling/Time taken: UnslothGRPOTrainer._calculate_rewards,▆▆▅▆▇▆▇▅▇▇▆█▅▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
profiling/Time taken: UnslothGRPOTrainer._prepare_inputs,▆▆▇█▇▇▇▇▆▅▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
profiling/Time taken: UnslothGRPOTrainer.answering_env_reward,▁▂█▁▁▂▁▂▁▁▁▂▁▂▁▁▁▁▁▁▁▁▁▁▁▂▁▁▁▁▂▂▁▂▁▁▁▁▁▁
profiling/Time taken: UnslothGRPOTrainer.answering_json_format_reward,▃▂▄▄▄▃▂█▃▂▂▄▃▂▂▁▂▃▄▂▂▃▂▃█▄▂▂▁▂▄▂▂▅▂▂▃▁▁▂
profiling/Time taken: UnslothGRPOTrainer.patronus_reward_fn,▇▇▇▆▇▆█▇▇▇▅▇█▆▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
profiling/Time taken: UnslothGRPOTrainer.persona_match_reward,▄▅▂██▅▇▃▇▄▄▆█▄▇▅▅▅▄▄▃▇▄▄▄▆▆▅▁▅▄▃▃▂▂▁█▄▂▅
profiling/Time taken: UnslothGRPOTrainer.vLLM.generate,▇█▇▇▇▇▇▇▇▇▁▆▄▇▄▆▅▆▆▆▆▆▆▆▆▅▆▆▆▆▄▂▆▃▆█▆▆▆▆
train/clip_ratio/high_max,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/clip_ratio/high_mean,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/clip_ratio/low_mean,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
+28,...


TrainOutput(global_step=192, training_loss=0.029096953990422964, metrics={'train_runtime': 7240.4456, 'train_samples_per_second': 0.027, 'train_steps_per_second': 0.027, 'total_flos': 0.0, 'train_loss': 0.029096953990422964})

In [10]:
# Save and push to Hub
output_dir = "./outputs/answering-grpo-final"
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)
print(f"Model saved to {output_dir}")

print(f"Pushing to Hub: {HF_REPO}")
trainer.push_to_hub(HF_REPO)
print(f"Model pushed to https://huggingface.co/{HF_REPO}")

Model saved to ./outputs/answering-grpo-final
Pushing to Hub: ricalanis/datasage-qwen-answering


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   0%|          |  554kB /  120MB            

  ...ering-grpo/tokenizer.json:  30%|###       | 3.46MB / 11.4MB            

  ...ng-grpo/training_args.bin:   1%|          |  73.0B / 7.38kB            

Model pushed to https://huggingface.co/ricalanis/datasage-qwen-answering
